In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Semantic search

In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("resources/acmecorp-employee-handbook.pdf")

data = loader.load()

print(data)

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-11-20T23:23:16+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-11-20T23:23:16+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'resources/acmecorp-employee-handbook.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Employee Handbook\nNon-Disclosure Agreement (NDA) Policy\nEmployees must protect confidential information belonging to the company, its clients, and partners.\nThis includes, but is not limited to, product roadmaps, customer data, internal communications,\nproprietary algorithms, financial information, and unreleased features. Confidential information may not\nbe shared with unauthorized individuals inside or outside the organization. These obligations continue\nafter employment ends.\nWorkplace Conduct Policy\nEmployees must maintain a respectful, professional environment

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(data)

print(len(all_splits))

3


Embedding Models: https://docs.langchain.com/oss/python/integrations/text_embedding

In [12]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="qwen3-embedding:latest")

In [13]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [14]:
ids = vector_store.add_documents(documents=all_splits)

In [22]:
query_text = "How many days of vacation does an employee get in their first year?"
single_vector = embeddings.embed_query(query_text)

results = vector_store.similarity_search(query_text)
print(str(single_vector)[:100])  # Show the first 100 characters of the vector
print(results[0])

[0.009527508, 0.015281198, -0.013139172, 0.001460776, 0.021268899, 0.013147083, -0.011100607, 0.0385
page_content='prohibited. Violations may result in disciplinary action.
Paid Time Off (PTO) Policy
Full■time employees accrue PTO according to the following schedule:  0–1 years of service: 10 days
per year (0.833 days per month)  1–3 years of service: 15 days per year (1.25 days per month)  3+
years of service: 20 days per year (1.67 days per month) PTO may be used for vacation, personal
needs, or illness. Requests should be submitted in advance through the HR system unless related to
an emergency. Employees may carry over up to 5 unused PTO days per calendar year. Extended
absences exceeding 5 consecutive business days require manager approval.
Travel & Expense Policy
Employees may be reimbursed for reasonable and necessary expenses incurred during approved
business travel. This includes transportation, lodging, meals, and incidental expenses within
established limits. Receipts mus

## RAG Agent

In [23]:
from langchain.tools import tool

@tool
def search_handbook(query: str) -> str:
    """Search the employee handbook for information"""
    results = vector_store.similarity_search(query)
    return results[0].page_content

In [27]:
from langchain.agents import create_agent

agent = create_agent(
    model="ollama:qwen3.5:latest",
    tools=[search_handbook],
    system_prompt="You are a helpful agent that can search the employee handbook for information."
    )

In [28]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many days of vacation does an employee get in their first year?")]}
)

In [29]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='How many days of vacation does an employee get in their first year?', additional_kwargs={}, response_metadata={}, id='5d6d76c8-31c7-4898-a921-52abe375aa91'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5:latest', 'created_at': '2026-04-07T11:23:02.044876Z', 'done': True, 'done_reason': 'stop', 'total_duration': 12950447583, 'load_duration': 4397133333, 'prompt_eval_count': 298, 'prompt_eval_duration': 3052138792, 'eval_count': 71, 'eval_duration': 5400714708, 'logprobs': None, 'model_name': 'qwen3.5:latest', 'model_provider': 'ollama'}, id='lc_run--019d67ae-4b84-7750-9d85-c02516e982ac-0', tool_calls=[{'name': 'search_handbook', 'args': {'query': 'vacation days first year employee'}, 'id': '6558dfe2-77f2-4154-98d2-ccbc3ff47052', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 298, 'output_tokens': 71, 'total_tokens': 369}),
              ToolMessage(content='prohibited. Vio